# 📘 Colab Notebook: Response/Output Evaluation Using Llumo

## 📝 Notebook Overview
This notebook helps you evaluate Output queries using Llumo’s powerful input-level metrics to ensure quality and safety:

### ✨ Metrics included:
  
- 🎯 Response Correctness
- 🧩 Response Completeness
- 🧠 Response Bias
- ☣️ Response Harmfulness
  
---

## 🚀 What you will do in this notebook:
- 📂 Load queries from the Excel file, get context according to the query, and get the output from OpenAI.  
- 🤖 Evaluate the queries for bias, correctness, completeness, and harmfulness  
- 📊 View the detailed evaluation results  
---
> 🔐 **Note:** The Llumo API key will be securely requested during runtime using Colab’s input prompt.


 ### **⚙️ Step 1: Install Dependencies**

In [27]:
!pip install llumo -q
!pip install openai -q

### **📂 Step 2: Import Required Libraries**

In [28]:
import pandas as pd
import getpass
import os
import requests

### **🔑 Setup OpenAI API Key & Llumo API key**

In [ ]:
import os

# Set your OpenAI API Key
os.environ["OPENAI_API_KEY"] = "Enter Your Open API Key"

# Set your Llumo API Key
os.environ["LLUMO_API_KEY"] = "Enter Your LLumo Key"

openai_key = os.getenv("OPENAI_API_KEY")
llumo_key = os.getenv("LLUMO_API_KEY")

### **🧾 Step 3: Load the Dataset - Optional**

In [33]:
# Make sure 'data.xlsx' is uploaded to your Colab environment
df = pd.read_excel("data.xlsx") # We have list of queries

# Preview the data
df.head()


,query
0,How can I return a laptop if I am not satisfie...
1,What do I do if my product arrives with a defect?
2,How does CyberShield handle a data breach?


### **List to store query results as dictionaries — `[{},{},{},{}]`**

This list collects the output of multiple queries run through the agent.  
Each query result is stored as a dictionary containing:

- `query`: The input question  
- `context`: The contextual data retrieved from the database or other sources to assist in answering the query  
- `output`: The LLM final response as plain text

The data used for evaluation will be in the following Example format:

```
[  
  {
    "query": "What is the capital of France?",
    "context": ["France is a country in Europe.", "Its capital city is Paris."],
    "output": "The capital of France is Paris."
  },
  {
    "query": "Summarize the plot of 'Romeo and Juliet'.",
    "context": ["'Romeo and Juliet' is a tragedy by William Shakespeare.", "It is about two lovers from feuding families."],
    "output": "Romeo and Juliet is a tragedy by William Shakespeare about two young lovers whose deaths ultimately reconcile their feuding families."
  }
]
```



In [39]:
queries = df["query"].to_list()

### **🔐 Step 4: Getting the context from Database**

In [40]:
import requests
from openai import OpenAI

# Function to get context from external API
def get_context(query):
    url = "https://model-api.llumo.ai/functionCalling/get-context-from-db"
    reqBody = {"query": query}
    response = requests.post(url, json=reqBody)
    return response.json()["contexts"]

# Get all contexts in a single API call
contexts = get_context(queries)

In [41]:
# Prepare list of dicts [{query, context, output}, ...]
results = []
for query, context in zip(queries, contexts):
    results.append({"query": query, "context": context, "output": ""})


### **🧾 Step 5: Generating the Outputs**


In [42]:
# Initialize OpenAI client
client = OpenAI(api_key=openai_key)

# ✅ Step 5: Generate LLM output for each item in the list
for item in results:
    query = item["query"]
    context = item["context"]

    response = client.chat.completions.create(
        model="gpt-4",
        messages=[
            {
                "role": "user",
                "content": f"Give answer to the given query: {query}, using the given context: {context}."
            }
        ],
        temperature=0.7
    )

    item["output"] = response.choices[0].message.content


#  Now 'results' is your final [{},{},{}] list with 'query', 'context', 'output'

### 📄 **Input Data with keys — "query", "context", "output"**
Preview the enriched data that will be passed for output evaluation.


In [43]:
results[0]

{'query': 'How can I return a laptop if I am not satisfied with it?',
 'context': 'ElectraTech is your go-to destination for the latest in technology and electronics. Our offerings include the newest smartphones, laptops, smart home devices, and more. Take advantage of free shipping on orders over $150. All products come with a one-year warranty covering manufacturing defects. Returns are accepted within 30 days, provided the item is in its original, unopened packaging. Our dedicated customer support team is available 24\\/7 to help with any questions or concerns related to products, returns, or warranty claims. At ElectraTech, we are committed to providing exceptional products and outstanding service.',
 'output': "If you are not satisfied with your laptop from ElectraTech, you can return it within 30 days. However, it's important to note that returns are only accepted if the item is in its original, unopened packaging. If you have any questions or concerns related to returns, you can

### 🤖 **Step 5: Initialize Llumo Client And Evaluate Output**
This block initializes the `LlumoClient` and evaluates the quality and safety of output using selected KPIs like:

- 🎯 Response Correctness
- 🧩 Response Completeness
- 🧠 Response Bias
- ☣️ Response Harmfulness

Additional Metrics:
- Context Utilization
- Hallucination
  


In [47]:

# Import the evaluation client from Llumo SDK
from llumo import LlumoClient

# Initialize the LlumoClient with your API key
client = LlumoClient(api_key = llumo_key)  # Replace with actual API key

resultDf = client.evaluateMultiple(
    data = results,  # Input Data
    evals = ["Response Completeness", "Response Correctness", "Response Bias","Context Utilization" ,"Hallucination"],  # Selected evaluation KPIs
    prompt_template = "Give answer to the given query: {{query}}, using the given context: {{context}}.",  # Prompt used for generation
    createExperiment = False,   # Set to True to save results as an experiment on the Llumo platform. If False, returns results as a DataFrame or A Python Dict. - Optional
    getDataFrame = True, # Return result as a DataFrame (True) or dictionary (False) - Optional
    )



Processing Batches: 100%|██████████| 5/5 [00:17<00:00,  3.49s/batch]


In [48]:
resultDf.head()

,query,context,output,Response Completeness,Response Completeness Reason,Response Correctness,Response Correctness Reason,Response Bias,Response Bias Reason,Context Utilization,Context Utilization Reason,Hallucination,Hallucination Reason
0,How can I return a laptop if I am not satisfie...,ElectraTech is your go-to destination for the ...,If you are not satisfied with your laptop from...,100,The response directly answers the query and in...,100,The response accurately reflects the return po...,1,The response provides customer service informa...,99,The response accurately reflects all relevant ...,21,"The output is a paraphrase of the context, co..."
1,What do I do if my product arrives with a defect?,FutureGadgets is your source for the most adva...,"If your product arrives with a defect, please ...",99,The response directly answers the query and in...,99,The response accurately reflects the context's...,2,The response provides instructions on product ...,100,"The response accurately reflects the warranty,...",20,The output is largely consistent with the cont...
2,How does CyberShield handle a data breach?,CyberShield Solutions is a premier provider of...,The context does not provide specific informat...,99,The response accurately reflects the context's...,100,The response accurately reflects the context's...,2,"The response is neutral and factual, lacking a...",100,The response accurately identifies the absence...,2,The output accurately reflects the context's l...
